# HASTIKA Task B on KaggleSix-way hate category over Kannada-English code-mixed comments, scored on macro-F1.**Before running:** in the right-hand sidebar set **Accelerator** to `GPU T4 x2` or`GPU P100`, and turn **Internet** on. Internet needs a phone-verified account; withoutit the clone and the model download both fail.Flip the switches in the next cell, then Run All. For anything past half an hour use**Save Version -> Save & Run All** rather than an interactive session, which dies withthe browser tab.

## 0. What to runTwo passes is the intended flow. First `RUN_ARMS = True` with `RUN_FULL = False` to findout which encoder wins on a cheap holdout. Then set `ENCODER` to the winner, flip to`RUN_ARMS = False` / `RUN_FULL = True`, and commit the long run.

In [ ]:
REPO   = "https://github.com/robinpnalex/Hastika-ICON2026.git"BRANCH = "task-b"RUN_FLOOR = True    # TF-IDF floor, CPU, seconds. Always worth it: it is the number to beat.RUN_SMOKE = True    # 300 rows, 1 epoch, ~2 min. Proves the GPU path before the long run.RUN_TAPT  = True    # masked-LM domain adaptation, ~10-15 minRUN_ARMS  = True    # three encoders on a 15% holdout, ~20 min eachRUN_FULL  = False   # the 5-fold submission run, 1.5-2 h# Used by the full run. After the arms finish, set this to whichever one won.ENCODER = "google/muril-base-cased"# ENCODER = "Hate-speech-CNERG/kannada-codemixed-abusive-MuRIL"# ENCODER = "work/runs/tapt-muril"SEEDS = "42"        # "42 43 44" averages three runs; costs 3x, worth it before believing                    # any gap under about one point

## 1. Clone and install

In [ ]:
import os, subprocess, sysWORK = "/kaggle/working/hastika"if os.path.isdir(WORK + "/.git"):    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)else:    subprocess.run(["git", "clone", "-q", "-b", BRANCH, "--depth", "1", REPO, WORK], check=True)os.chdir(WORK)print("cwd:", os.getcwd())

In [ ]:
!pip install -q emoji ftfy "transformers>=4.45,<6"import torch, transformersprint("torch", torch.__version__, "| transformers", transformers.__version__)if torch.cuda.is_available():    print("gpu:", torch.cuda.get_device_name(0),          "| bf16:", torch.cuda.is_bf16_supported(), "(fp16 is used when this is False)")else:    print("NO GPU -- set Accelerator in the sidebar, everything below will crawl")

## 2. The floorTF-IDF character n-grams and a linear SVM, on the same folds the transformer uses.Expect `macro-F1 0.5948`. Anything the GPU produces has to beat this to be worth its hour.

In [ ]:
if RUN_FLOOR:    !python work/baseline_svm.py --task b --demojize

## 3. Smoke testTwo minutes. You are checking that the device line names your GPU, that the batch sizefits, and that the run reaches `wrote .../predictions.csv`. The score here is noise.

In [ ]:
if RUN_SMOKE:    !python -u work/muril_b.py --tag b_smoke --folds 0 --epochs 1 --limit 300

## 4. Domain-adapt MuRILA masked-LM pass over the training rows plus the external Kannada corpus. MuRIL spends2.28 wordpieces per whitespace word on this register, which is what this tries to fix.It reads no labels, and it refuses Task A's files: 319 of the 395 Task B test ids alsoappear in `binary_train.csv`.Watch the held-out perplexity. If it does not fall substantially, drop the `b_tapt` arm.

In [ ]:
if RUN_TAPT:    !python -u work/tapt.py --out work/runs/tapt-muril 2>&1 | tee work/tapt.log

## 5. Rank the encodersThree arms on a 15% holdout, about a fifth the cost of a full run each. The second is aMuRIL already fine-tuned on code-mixed Kannada abusive speech, with a vocabularybyte-identical to stock MuRIL.Each arm prints two scores. **Compare arms on the `last` number.** The `best` number ischosen on the rows it reports, so it flatters every arm by a different amount.

In [ ]:
if RUN_ARMS:    !python -u work/muril_b.py --tag b_base    --folds 0 --seeds $SEEDS 2>&1 | tee work/b_base.log    !python -u work/muril_b.py --tag b_abusive --folds 0 --seeds $SEEDS --model Hate-speech-CNERG/kannada-codemixed-abusive-MuRIL 2>&1 | tee work/b_abusive.log    import os    if os.path.isdir("work/runs/tapt-muril"):        !python -u work/muril_b.py --tag b_tapt --folds 0 --seeds $SEEDS --model work/runs/tapt-muril 2>&1 | tee work/b_tapt.log

In [ ]:
if RUN_ARMS:    !grep -H "holdout macro-F1" work/b_*.log

## 6. The submission runFive folds, six epochs. Set `ENCODER` above to whichever arm won before running this.

In [ ]:
if RUN_FULL:    !python -u work/muril_b.py --tag muril_b --model $ENCODER --seeds $SEEDS 2>&1 | tee work/muril_b.log

## 7. Blend and packageThe blender takes every five-fold run in `work/runs/` whose shape matches Task B, so theholdout arms are skipped automatically and Task A runs can never be mixed in. If theblend does not beat the single model, package `work/runs/muril_b/predictions.csv` instead.

In [ ]:
if RUN_FULL:    !python work/ensemble.py --task b

In [ ]:
if RUN_FULL:    SUBMIT = "work/runs/ensemble_b/predictions.csv"   # or work/runs/muril_b/predictions.csv    !python work/make_submission.py --task b --pred $SUBMIT --out /kaggle/working/task_b_predictions.zip    !cp work/muril_b.log /kaggle/working/ 2>/dev/null    !ls -la /kaggle/working/

## 8. Getting it outOpen the version's **Output** tab and download `task_b_predictions.zip`, then upload itto the Task B phase on CodaBench. Take `muril_b.log` too if you want the per-class reportand confusion matrix later.Only `/kaggle/working` reaches that tab. Model weights cache elsewhere and do not countagainst the 20 GB output cap.**If you run out of memory**, in order of how much they buy: `--bs 8 --grad-accum 2`,then `--no-fgm`, then `--max-len 128`. Ignore `--trim-vocab`, which exists for a smallCPU box.**Budget:** 12 hours per session, 30 GPU-hours a week.